# Kubernetes 第4周：生产实践

> **学习目标**：掌握日志管理、监控告警、排障技巧、CI/CD 集成、安全加固

---

## 日志管理

在 K8s 中，日志管理有三层：

```
应用 ──stdout/stderr──▶ Docker/containerd ──▶ 节点上的日志文件
                                                │
                              ┌─────────────────┘
                              ▼
                    日志收集器（Fluentd / Filebeat）
                              │
                              ▼
                    日志存储（Elasticsearch / Loki）
                              │
                              ▼
                    可视化（Kibana / Grafana）
```

### kubectl logs 的边界

`kubectl logs` 只能看**单个 Pod 的一个容器**的日志。Pod 被删除后日志就没了。
所以生产环境必须部署日志收集系统。

### 最佳实践：结构化日志

让你的 Python 应用输出 JSON 格式日志，而不是纯文本：

In [ ]:
! mkdir -p /tmp/k8s-demo

%%writefile /tmp/k8s-demo/logging_demo.py
import logging
import json
import sys
from datetime import datetime, timezone

# 结构化日志：用 JSON 格式输出到 stdout
class JSONFormatter(logging.Formatter):
    def format(self, record):
        log_entry = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
            "module": record.module,
            "function": record.funcName,
        }
        if record.exc_info and record.exc_info[0]:
            log_entry["exception"] = self.formatException(record.exc_info)
        return json.dumps(log_entry, ensure_ascii=False)

# 配置
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(JSONFormatter())
logger = logging.getLogger("myapp")
logger.addHandler(handler)
logger.setLevel(logging.INFO)

# 模拟
logger.info("用户登录", extra={"user_id": 1001})
logger.warning("访问频率过高", extra={"ip": "10.0.0.5"})
try:
    1 / 0
except:
    logger.error("计算错误", exc_info=True)

print("\n↑ JSON 格式日志方便 Logstash/Fluentd 解析和检索")

### 常用日志方案对比

| 方案 | 组件 | 适用场景 |
|------|------|----------|
| **EFK** | Elasticsearch + Fluentd + Kibana | 大集群、全文检索 |
| **PLG** | Promtail + Loki + Grafana | 轻量、和 Prometheus 集成好 |
| **云原生** | CloudWatch / Stackdriver / Azure Monitor | 上云就用云的 |

---

## 监控与告警

### Prometheus + Grafana：事实标准

```
Prometheus（拉取指标）──▶ 存储时序数据
    │
    ├── 自动发现 K8s Pod/Service（通过 ServiceMonitor）
    ├── 评估告警规则 → AlertManager → 通知（Slack/邮件/PagerDuty）
    └── Grafana 读取数据 → 可视化面板
```

### 关键指标

| 类别 | 指标 | 告警条件 |
|------|------|----------|
| **Pod 状态** | `kube_pod_status_phase` | Pod 不是 Running |
| **资源使用** | `container_cpu_usage_seconds_total` | CPU > limits 的 80% |
| **内存** | `container_memory_working_set_bytes` | 内存 > limits 的 80% |
| **重启** | `kube_pod_container_status_restarts_total` | 短时间内大量重启 |
| **请求延迟** | `http_request_duration_seconds` | P99 > 阈值 |
| **错误率** | `http_requests_total{status=~"5.."}` | 5xx 占比 > 5% |

In [ ]:
# 安装 Prometheus Stack（kube-prometheus-stack）
print("在 K8s 中安装监控栈：")
print()
print("# 添加 Helm repo")
print("helm repo add prometheus-community https://prometheus-community.github.io/helm-charts")
print()
print("# 安装 kube-prometheus-stack（含 Prometheus + Grafana + AlertManager）")
print("helm install monitoring prometheus-community/kube-prometheus-stack \\")
print("  --namespace monitoring --create-namespace \\")
print("  --set grafana.adminPassword=admin")
print()
print("# 获取 Grafana 密码")
print("kubectl get secret monitoring-grafana -n monitoring -o jsonpath='{.data.admin-password}' | base64 -d")
print()
print("# 端口转发访问 Grafana")
print("kubectl port-forward -n monitoring svc/monitoring-grafana 3000:80")
print("# 浏览器打开 http://localhost:3000")

---

## 排障实战

K8s 排障遵循一个套路：**自底向上，逐层排查。**

```
1. Pod 在吗？         → kubectl get pods
2. Pod 状态是什么？    → kubectl describe pod（看 Events）
3. 容器日志说什么？    → kubectl logs
4. 服务通吗？         → kubectl get svc + endpoints
5. 网络策略拦了吗？    → kubectl get networkpolicies
6. 节点资源够吗？      → kubectl top nodes
7. 进入容器内部排查    → kubectl exec -it <pod> -- bash
```

In [ ]:
# 故意制造故障，然后排查
%%writefile /tmp/k8s-demo/bug-deploy.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: bug-app
spec:
  replicas: 1
  selector:
    matchLabels:
      app: buggy
  template:
    metadata:
      labels:
        app: buggy
    spec:
      containers:
      - name: app
        image: python:3.12-slim
        command: ["python", "-c"]
        args: ["import time; time.sleep(5); raise SystemExit(1)"]  # Bug: 5秒后崩溃
      restartPolicy: Always

! kubectl apply -f /tmp/k8s-demo/bug-deploy.yaml 2>/dev/null
! sleep 10
print("=== 排查开始 ===")
! kubectl get pods -l app=buggy 2>/dev/null
! echo "---"
! kubectl describe pod -l app=buggy 2>/dev/null | tail -15
! echo "---"
! kubectl logs -l app=buggy --previous 2>/dev/null || kubectl logs -l app=buggy 2>/dev/null

In [ ]:
print("排查结论：")
print("  Pod 不断重启（CrashLoopBackOff）")
print("  Events 显示: Back-off restarting failed container")
print("  Logs 显示: SystemExit(1) — 应用启动后 5 秒崩溃")
print("  修复：改正代码逻辑")

# 清理
! kubectl delete deployment bug-app --wait=false 2>/dev/null

### 排障速查表

| 症状 | 首选命令 | 常见原因 |
|------|----------|----------|
| Pod Pending | `describe pod` | 资源不够、PVC 未绑定、镜像拉取慢 |
| Pod CrashLoopBackOff | `logs` / `describe` | 应用启动失败、健康检查没过 |
| Pod ImagePullBackOff | `describe pod` | 镜像名写错、Registry 连不上 |
| 服务访问不通 | `get endpoints` | Selector 不匹配、targetPort 不对 |
| Pod 频繁重启 | `logs --previous` | OOMKilled（看 Events）、应用 Bug |
| 性能慢 | `top` / `exec` 进容器 | 资源 limits 太低导致 throttle |

---

## CI/CD 集成：GitOps

### 从手动到自动化

```
传统部署：                      GitOps：
开发 commit                   开发 commit
  ↓                              ↓
CI: 构建镜像                    CI: 构建镜像 + 推送
  ↓                              ↓
人工 kubectl apply             Git 仓库（部署配置）
  ↓                              ↓
人工验证                       ArgoCD / Flux（自动同步）
                                 ↓
                               K8s 集群
```

GitOps 的核心思想：**Git 仓库里存的就是集群的期望状态，工具自动把集群同步到 Git 的状态。**

In [ ]:
# GitHub Actions + K8s 部署示例
%%writefile /tmp/k8s-demo/ci-example.yaml
# .github/workflows/deploy.yml 示例
# 演示目的，注释多过代码

print("CI/CD 流水线模板：")
print()
print("name: Build and Deploy")
print("on:")
print("  push:")
print("    branches: [main]")
print()
print("jobs:")
print("  build-and-deploy:")
print("    runs-on: ubuntu-latest")
print("    steps:")
print("      # 1. 检出代码")
print("      - uses: actions/checkout@v4")
print()
print("      # 2. 构建 Docker 镜像并推送到 Registry")
print("      - uses: docker/build-push-action@v5")
print("        with:")
print("          push: true")
print("          tags: registry.example.com/app:${{ github.sha }}")
print()
print("      # 3. 更新 K8s 部署的镜像版本")
print("      - run: |")
print("          kubectl set image deployment/app \\")
print("            app=registry.example.com/app:${{ github.sha }}")
print("          kubectl rollout status deployment/app")
print()
print("# 更专业的方案：用 ArgoCD 或 Flux 实现 GitOps")

---

## 安全最佳实践

### 检查清单

| # | 检查项 | 做法 |
|---|--------|------|
| 1 | 非 root 运行 | `securityContext.runAsNonRoot: true` |
| 2 | 只读根文件系统 | `securityContext.readOnlyRootFilesystem: true` |
| 3 | 禁止 privilege escalation | `securityContext.allowPrivilegeEscalation: false` |
| 4 | 镜像扫描 | CI 中集成 Trivy 扫描漏洞 |
| 5 | NetworkPolicy | 只允许必要的 Pod 间流量 |
| 6 | Secret 加密 | 用 Sealed Secrets 或 External Secrets，别裸用 base64 |
| 7 | RBAC 最小权限 | 每个 ServiceAccount 只给必需的权限 |
| 8 | Pod Security Standards | 启用 `baseline` 或 `restricted` 策略 |
| 9 | 资源限制 | 所有容器都设 requests/limits |
| 10 | 镜像固定版本 | 不用 `latest` tag，用具体版本号或 digest |

In [ ]:
%%writefile /tmp/k8s-demo/secure-pod.yaml
# 安全 Pod 配置模板
apiVersion: v1
kind: Pod
metadata:
  name: secure-demo
spec:
  securityContext:
    runAsNonRoot: true            # 必须以非 root 运行
    runAsUser: 1000               # 指定 UID
    fsGroup: 1000                 # 文件系统组
  containers:
  - name: app
    image: python:3.12-slim
    securityContext:
      allowPrivilegeEscalation: false   # 禁止提权
      readOnlyRootFilesystem: true      # 根文件系统只读
      capabilities:
        drop: ["ALL"]                    # 去掉所有能力
    resources:
      requests:
        cpu: "100m"
        memory: "128Mi"
      limits:
        cpu: "200m"
        memory: "256Mi"

print("安全配置要点：")
print("  - 非 root 运行 (runAsNonRoot)")
print("  - 禁止提权 (allowPrivilegeEscalation: false)")
print("  - 只读根文件系统 (readOnlyRootFilesystem)")
print("  - 去掉所有 Linux capabilities")
print("  - 必须设资源限制")

---

## 核心命令速查表

这是你日常 90% 时间会用到的命令：

| 操作 | 命令 |
|------|------|
| 查看资源 | `kubectl get <kind>` / `kubectl get all` |
| 详细信息 | `kubectl describe <kind> <name>` |
| 创建/更新 | `kubectl apply -f <file>` |
| 删除 | `kubectl delete <kind> <name>` / `kubectl delete -f <file>` |
| 日志 | `kubectl logs <pod> [-c <container>] [-f] [--previous]` |
| 进入容器 | `kubectl exec -it <pod> [-- bash]` |
| 端口转发 | `kubectl port-forward <pod/svc> <local>:<remote>` |
| 扩缩容 | `kubectl scale deploy/<name> --replicas=N` |
| 滚动重启 | `kubectl rollout restart deploy/<name>` |
| 回滚 | `kubectl rollout undo deploy/<name>` |
| 资源使用 | `kubectl top pods` / `kubectl top nodes` |
| 事件 | `kubectl get events --sort-by=.metadata.creationTimestamp` |
| YAML 导出 | `kubectl get <kind> <name> -o yaml` |
| 解释文档 | `kubectl explain <kind>` / `kubectl explain <kind>.spec` |
| 上下文切换 | `kubectl config use-context <name>` |
| 命名空间 | `-n <namespace>` 或 `kubens` |

---

## 🎯 第4周总结 & 全课程回顾

### 第4周要点

| 主题 | 核心 |
|------|------|
| **日志** | 输出 stdout/stderr → 收集器 → 存储 → 可视化 |
| **监控** | Prometheus + Grafana：Pod 状态、资源、延迟、错误率 |
| **排障** | 自底向上：Pod → 日志 → Service → 网络 → 节点 |
| **CI/CD** | GitOps：Git 为单一事实来源，ArgoCD/Flux 自动同步 |
| **安全** | 最小权限、非 root、只读根 FS、NetworkPolicy、镜像扫描 |

### 四周学习回顾

```
Week 1: Pod / Deployment / Service / ConfigMap / Secret / Namespace
         "理解 K8s 怎么管理容器"
    ↓
Week 2: Ingress / PV-PVC / StatefulSet / NetworkPolicy / DNS
         "理解网络怎么通、数据怎么存"
    ↓
Week 3: 调度 / 资源管理 / 健康检查 / HPA / Helm / RBAC
         "理解怎么运维和自动化"
    ↓
Week 4: 日志 / 监控 / 排障 / CI/CD / 安全
         "生产环境生存技能"
```

---

## 🧪 最终项目：生产级 K8s 部署

从零部署一个完整的 Python 博客系统：

```
要求：
├── Nginx（2 副本，HPA）— 前端
├── FastAPI（3 副本，HPA）— API 服务
├── Celery Worker（2 副本）— 异步任务
├── PostgreSQL（StatefulSet，1 副本）— 数据库
├── Redis（1 副本）— 缓存 + 消息队列
├── Ingress 配置域名 + TLS
├── Prometheus 监控 + Grafana 面板
├── 日志收集（Fluentd 或 Loki）
├── 完整 Helm Chart
├── RBAC + NetworkPolicy + 安全配置
└── CI/CD 流水线使：Git Push → 自动部署
```

> 这个项目综合了全部四周的知识点。完成后你会对"如何在生产环境跑 K8s"有完整认知。

In [ ]:
# 你的最终项目
pass